# RESPOND Political Corruption Corpus Workflow

Run this notebook top to bottom when rebuilding the main analysis data.

The workflow is:

1. Load the raw country CSVs from Research Drive.
2. Inspect raw country, language, and source counts.
3. Fix text encoding issues where needed.
4. Clean and deduplicate the corpus into `df_clean`.
5. Save compressed cleaned files to the 1 TB storage disk.
6. Build denominator tables from the cleaned corruption-query corpus.
7. Load the 452 manual labels.
8. Evaluate a TF-IDF baseline and a multilingual embedding classifier.
9. Build an active-learning batch for additional annotation.

Important: the raw files were collected using corruption-related search terms, so `df_clean` is a cleaned corruption-query corpus, not a full all-news denominator.


In [1]:
from pathlib import Path
import hashlib
import os
import re
import warnings

import pandas as pd
from pandas.errors import DtypeWarning

from config import RD_BASE_DIR
from dataloader import load_country_news_files_webdav, load_human_annotated_for_translation_webdav

warnings.filterwarnings("ignore", category=DtypeWarning)


In [2]:
NEWS_DIR = RD_BASE_DIR
COUNTRIES = None  # None means: discover and load every *_news.csv file.
MIN_WORDS = 80
RANDOM_STATE = 42

# Large storage location on annecuda. Override by setting RESPOND_OUTPUT_DIR if needed.
DEDUP_OUTPUT_DIR = Path(
    os.environ.get(
        "RESPOND_OUTPUT_DIR",
        "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline",
    )
)
DEDUP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Research Drive input directory: {NEWS_DIR}")
print(f"Output directory:              {DEDUP_OUTPUT_DIR}")


Research Drive input directory: ASCOR-FMG-5580-RESPOND-news-data (Projectfolder)
Output directory:              /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline


In [3]:
df = load_country_news_files_webdav(data_dir=NEWS_DIR, countries=COUNTRIES)
print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")
df.head()

Loaded 3,092,051 rows and 26 columns.


,Unnamed: 0,uri,lang,isDuplicate,date,time,dateTime,dateTimePub,dataType,sim,...,sentiment,wgt,relevance,source.uri,source.dataType,source.title,country,combined_text,userHasPermissions,source.private
0,0,8285618774,bul,False,2024-08-21,22:43:38,2024-08-21T22:43:38Z,2024-08-21T22:42:15Z,news,0.0,...,NaN,102,102,defakto.bg,news,DeFakto.bg,Bulgaria,"Отново осъдиха ВАС и НАП за ""съществено"" наруш...",NaN,NaN
1,1,7710446361,bul,True,2023-09-04,11:15:29,2023-09-04T11:15:29Z,2023-09-04T11:13:12Z,news,0.0,...,NaN,102,102,plovdiv24.bg,news,Plovdiv24.bg,Bulgaria,"Магазините стават банкомати, няма да е необход...",NaN,NaN
2,2,7710431035,bul,True,2023-09-04,11:06:38,2023-09-04T11:06:38Z,2023-09-04T11:05:10Z,news,0.0,...,NaN,102,102,burgas24.bg,news,Burgas24.bg,Bulgaria,"Магазините стават банкомати, няма да е необход...",NaN,NaN
3,3,7710419617,bul,False,2023-09-04,10:59:24,2023-09-04T10:59:24Z,2023-09-04T10:54:52Z,news,0.0,...,NaN,102,102,varna24.bg,news,Varna24.bg,Bulgaria,"Магазините стават банкомати, няма да е необход...",NaN,NaN
4,4,7710419595,bul,True,2023-09-04,10:59:17,2023-09-04T10:59:17Z,2023-09-04T10:57:31Z,news,0.0,...,NaN,102,102,ruse24.bg,news,Ruse24.bg,Bulgaria,"Магазините стават банкомати, няма да е необход...",NaN,NaN


In [4]:
print("Country counts:")
display(df["country"].value_counts(dropna=False).to_frame("count"))

if "lang" in df.columns:
    print("Language counts:")
    display(df["lang"].value_counts(dropna=False).to_frame("count"))

if "source.uri" in df.columns:
    print("Top source.uri values:")
    display(df["source.uri"].value_counts(dropna=False).head(20).to_frame("count"))

Country counts:


,count
country,
United_Kingdom,1039569
Italy,910942
France,412707
Bulgaria,260300
Hungary,144338
Netherlands,121188
Ukraine,103284
Sweden,60598
Serbia,39125


Language counts:


,count
lang,
eng,1041774
ita,910637
fra,412710
bul,260258
hun,144334
nld,119268
ukr,102636
swe,60533
srp,39129


Top source.uri values:


,count
source.uri,
dailymail.co.uk,128090
reuters.com,37919
independent.co.uk,37732
theguardian.com,37306
ansa.it,25203
journalstar.com,20273
ft.com,19678
thesun.co.uk,16403
ilgiornale.it,15635


## Helpers

`fix_mojibake` repairs text like `Ð¡Ð»ÑÐ¶Ð±Ð°...` back into readable Cyrillic, and also repairs common `Ã¡`-style Latin accent damage. If text is already fine, it is returned unchanged.


In [18]:
TEXT_COLUMN_CANDIDATES = [
    "translated_text",
    "combined_text",
    "body",
    "text",
    "article_text",
    "content",
]


KEEP_CLEANED_COLUMNS = [
    "uri",
    "country",
    "dateTime",
    "dateTimePub",
    "date",
    "date_parsed",
    "year",
    "month",
    "week",
    "source_uri",
    "article_text",
    "word_count",
    "text_hash",
    "near_dup_hash",
]

MINIMAL_COMBINED_COLUMNS = [
    "uri",
    "country",
    "dateTime",
    "date_parsed",
    "year",
    "month",
    "week",
    "source_uri",
    "word_count",
    "text_hash",
    "near_dup_hash",
]


def choose_text_column(dataframe):
    for column in TEXT_COLUMN_CANDIDATES:
        if column in dataframe.columns:
            return column
    raise ValueError("No usable text column found. Expected one of: " + ", ".join(TEXT_COLUMN_CANDIDATES))


def fix_mojibake(text):
    if not isinstance(text, str):
        return ""

    candidates = [text]

    # Common case: UTF-8 decoded as Latin-1.
    try:
        candidates.append(text.encode("latin1").decode("utf-8"))
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    # Common case: UTF-8 decoded as Windows-1252.
    try:
        candidates.append(text.encode("cp1252").decode("utf-8"))
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    def badness(s):
        markers = ["Ð", "Ñ", "Ã", "Â", "Ä", "Å", "�"]
        return sum(s.count(m) for m in markers)

    return min(candidates, key=badness)


def normalize_text(text):
    text = fix_mojibake(text)
    text = text.replace(" ", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_for_hash(text):
    text = normalize_text(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def text_hash(text):
    normalized = normalize_for_hash(text)
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()


def cheap_near_duplicate_fingerprint(text, n_tokens=80):
    normalized = normalize_for_hash(text)
    fingerprint = " ".join(normalized.split()[:n_tokens])
    return hashlib.md5(fingerprint.encode("utf-8")).hexdigest()


## Clean And Deduplicate

This creates the denominator dataset for the corruption-query corpus.

Cleaning steps:

1. Select the best available article text field. In this dataset this is usually `combined_text`.
2. Repair common mojibake/encoding damage.
3. Drop articles flagged as duplicates by the source data.
4. Drop empty or very short articles.
5. Parse dates and add year, month, and week fields.
6. Deduplicate by `uri`, exact normalized text, and a simple near-duplicate fingerprint.


In [7]:
df_clean = df.copy()
print(f"Starting rows: {len(df_clean):,}")

text_col = choose_text_column(df_clean)
print(f"Using text column: {text_col}")
df_clean["article_text"] = df_clean[text_col].map(normalize_text)

if "isDuplicate" in df_clean.columns:
    before = len(df_clean)
    is_duplicate = df_clean["isDuplicate"].astype(str).str.lower().isin(["true", "1", "yes"])
    df_clean = df_clean[~is_duplicate].copy()
    print(f"After dropping isDuplicate=True rows: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean["word_count"] = df_clean["article_text"].str.split().str.len().fillna(0).astype(int)
df_clean = df_clean[df_clean["article_text"].ne("")].copy()
print(f"After removing missing/empty text: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean = df_clean[df_clean["word_count"] >= MIN_WORDS].copy()
print(f"After removing articles with word_count < {MIN_WORDS}: {len(df_clean):,} (-{before - len(df_clean):,})")

if "dateTime" in df_clean.columns:
    date_source = df_clean["dateTime"]
elif "dateTimePub" in df_clean.columns:
    date_source = df_clean["dateTimePub"]
elif "date" in df_clean.columns:
    date_source = df_clean["date"]
else:
    date_source = pd.Series(pd.NaT, index=df_clean.index)

df_clean["date_parsed"] = pd.to_datetime(date_source, errors="coerce", utc=True)
date_naive = df_clean["date_parsed"].dt.tz_convert(None)
df_clean["year"] = date_naive.dt.year.astype("Int64")
df_clean["month"] = date_naive.dt.to_period("M").dt.to_timestamp()
df_clean["week"] = date_naive.dt.to_period("W").dt.start_time

if "source.uri" in df_clean.columns:
    df_clean["source_uri"] = df_clean["source.uri"]
elif "source" in df_clean.columns:
    df_clean["source_uri"] = df_clean["source"]
else:
    df_clean["source_uri"] = None

if "uri" in df_clean.columns:
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=["uri"], keep="first").copy()
    print(f"After URI dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean["text_hash"] = df_clean["article_text"].map(text_hash)
df_clean = df_clean.drop_duplicates(subset=["text_hash"], keep="first").copy()
print(f"After exact text dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean["near_dup_hash"] = df_clean["article_text"].map(cheap_near_duplicate_fingerprint)
df_clean = df_clean.drop_duplicates(subset=["near_dup_hash"], keep="first").copy()
print(f"After cheap near-duplicate dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

display(df_clean.head())

Starting rows: 3,092,051
Using text column: combined_text
After dropping isDuplicate=True rows: 2,328,641 (-763,410)
After removing missing/empty text: 2,328,641 (-0)
After removing articles with word_count < 80: 2,301,885 (-26,756)
After URI dedupe: 2,301,885 (-0)
After exact text dedupe: 2,278,760 (-23,125)
After cheap near-duplicate dedupe: 2,264,916 (-13,844)


,Unnamed: 0,uri,lang,isDuplicate,date,time,dateTime,dateTimePub,dataType,sim,...,source.private,article_text,word_count,date_parsed,year,month,week,source_uri,text_hash,near_dup_hash
0,0,8285618774,bul,False,2024-08-21,22:43:38,2024-08-21T22:43:38Z,2024-08-21T22:42:15Z,news,0.000000,...,NaN,"Отново осъдиха ВАС и НАП за ""съществено"" наруш...",3874,2024-08-21 22:43:38+00:00,2024,2024-08-01,2024-08-19,defakto.bg,1ce00647778ad5015afe62199b2d3b0b,0fdf86297792cf0fbc6d68bec1e29c2b
3,3,7710419617,bul,False,2023-09-04,10:59:24,2023-09-04T10:59:24Z,2023-09-04T10:54:52Z,news,0.000000,...,NaN,"Магазините стават банкомати, няма да е необход...",1706,2023-09-04 10:59:24+00:00,2023,2023-09-01,2023-09-04,varna24.bg,ea5be2da0d442e48a25d2e77aabf4d56,c90eefeee27bf501246acaab6d31f436
5,5,7709882307,bul,False,2023-09-04,05:46:39,2023-09-04T05:46:39Z,2023-09-04T05:30:00Z,news,0.000000,...,NaN,"Източват банковата ни сметка със ""спуфинг"" - Т...",1789,2023-09-04 05:46:39+00:00,2023,2023-09-01,2023-09-04,trud.bg,f62e779ef4ecc321113f9ed78d7d99cf,0ad766317db3e50848037fcedbdb646b
6,6,7194252691,bul,False,2022-09-19,19:02:00,2022-09-19T19:02:00Z,2022-09-19T18:34:00Z,news,0.000000,...,NaN,Честните нови играчи в политиката бяха измама ...,366,2022-09-19 19:02:00+00:00,2022,2022-09-01,2022-09-19,trud.bg,ce4d7428f985005d5db1b5cc9bbf75a1,16ba930de89c18dc05ae407a4fa8a6ab
7,7,6588654023,bul,False,2021-06-02,18:03:00,2021-06-02T18:03:00Z,2021-06-02T18:02:00Z,news,0.819608,...,NaN,Пеевски и Черепа в черния списък на САЩ Служба...,2924,2021-06-02 18:03:00+00:00,2021,2021-06-01,2021-05-31,glasove.com,ad02b07489ea1e8d147f5cfa900db8ec,ade51726e1191415a17e7f8523c8558e


In [8]:
denom_country_total = (
    df_clean.groupby("country", dropna=False)
    .size()
    .reset_index(name="total_articles")
    .sort_values("total_articles", ascending=False)
)

display(denom_country_total)
print(f"Total cleaned/deduped rows: {denom_country_total['total_articles'].sum():,}")

,country,total_articles
3,Italy,736203
8,United_Kingdom,662414
1,France,290206
0,Bulgaria,194416
2,Hungary,104278
4,Netherlands,100789
7,Ukraine,97079
6,Sweden,45640
5,Serbia,33891


Total cleaned/deduped rows: 2,264,916


## Save Cleaned Outputs

This saves a minimal combined file and compressed full-text per-country files to the 1 TB disk. It avoids a huge uncompressed combined full-text CSV.


In [9]:
keep_columns = [column for column in KEEP_CLEANED_COLUMNS if column in df_clean.columns]
minimal_columns = [column for column in MINIMAL_COMBINED_COLUMNS if column in df_clean.columns]

combined_output = DEDUP_OUTPUT_DIR / "all_countries_cleaned_deduped_minimal.csv.gz"
df_clean[minimal_columns].to_csv(combined_output, index=False, compression="gzip")
print(f"Saved minimal combined cleaned file: {combined_output}")

for country, country_df in df_clean.groupby("country", dropna=False):
    safe_country = str(country).replace("/", "_")
    country_output = DEDUP_OUTPUT_DIR / f"{safe_country}_cleaned_deduped.csv.gz"
    country_df[keep_columns].to_csv(country_output, index=False, compression="gzip")
    print(f"Saved {len(country_df):,} rows: {country_output}")

Saved minimal combined cleaned file: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/all_countries_cleaned_deduped_minimal.csv.gz
Saved 194,416 rows: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/Bulgaria_cleaned_deduped.csv.gz
Saved 290,206 rows: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/France_cleaned_deduped.csv.gz
Saved 104,278 rows: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/Hungary_cleaned_deduped.csv.gz
Saved 736,203 rows: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/Italy_cleaned_deduped.csv.gz
Saved 100,789 rows: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/Netherlands_cleaned_deduped.csv.gz
Saved 33,891 rows: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/Serbia_cleaned_dedu

## Build Denominator Tables

These are the denominator counts for the cleaned corruption-query corpus.


In [10]:
denom_country_year = (
    df_clean.groupby(["country", "year"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

denom_country_month = (
    df_clean.groupby(["country", "month"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

denom_country_week = (
    df_clean.groupby(["country", "week"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

denom_country_year_source = (
    df_clean.groupby(["country", "year", "source_uri"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

print("Country-year:")
display(denom_country_year.head(20))
print("Country-month:")
display(denom_country_month.head(20))
print("Country-week:")
display(denom_country_week.head(20))


Country-year:


,country,year,total_articles
0,Bulgaria,2018,13598
1,Bulgaria,2019,21028
2,Bulgaria,2020,30023
3,Bulgaria,2021,32464
4,Bulgaria,2022,30236
5,Bulgaria,2023,29227
6,Bulgaria,2024,37840
7,France,2018,37467
8,France,2019,46288
9,France,2020,38545


Country-month:


,country,month,total_articles
0,Bulgaria,2018-01-01,669
1,Bulgaria,2018-02-01,1073
2,Bulgaria,2018-03-01,1009
3,Bulgaria,2018-04-01,1321
4,Bulgaria,2018-05-01,1101
5,Bulgaria,2018-06-01,1024
6,Bulgaria,2018-07-01,869
7,Bulgaria,2018-08-01,1163
8,Bulgaria,2018-09-01,1410
9,Bulgaria,2018-10-01,1276


Country-week:


,country,week,total_articles
0,Bulgaria,2018-01-01,119
1,Bulgaria,2018-01-08,145
2,Bulgaria,2018-01-15,129
3,Bulgaria,2018-01-22,171
4,Bulgaria,2018-01-29,183
5,Bulgaria,2018-02-05,295
6,Bulgaria,2018-02-12,267
7,Bulgaria,2018-02-19,288
8,Bulgaria,2018-02-26,243
9,Bulgaria,2018-03-05,203


In [11]:
denom_country_year.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_year.csv", index=False)
denom_country_month.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_month.csv", index=False)
denom_country_week.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_week.csv", index=False)
denom_country_year_source.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_year_source.csv", index=False)

print(f"Saved denominator tables to: {DEDUP_OUTPUT_DIR}")

Saved denominator tables to: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline


## Manual Labels And Model Checks

The manually labelled validation set has 452 rows. The TF-IDF model is a simple baseline. The multilingual embedding model is currently more useful for active learning, but still not strong enough to treat as final labels without additional human review.


In [19]:
df_annotations = load_human_annotated_for_translation_webdav()
print(f"Loaded {len(df_annotations):,} annotated rows.")

print("Label counts:")
display(df_annotations["corruption_label_m"].value_counts(dropna=False))

print("Country counts:")
display(df_annotations["country"].value_counts(dropna=False))

Loaded 452 annotated rows.
Label counts:


corruption_label_m
no political corruption    332
political corruption       120
Name: count, dtype: int64

Country counts:


country
Serbia            52
United_Kingdom    51
Hungary           51
Netherlands       51
Italy             51
Sweden            51
France            49
Bulgaria          48
Ukraine           48
Name: count, dtype: int64

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

manual_df = df_annotations.copy()
manual_df["label_clean"] = manual_df["corruption_label_m"].astype(str).str.strip().str.lower()
manual_df = manual_df[manual_df["label_clean"].isin(["political corruption", "no political corruption"])].copy()
manual_df["y"] = manual_df["label_clean"].map({"political corruption": 1, "no political corruption": 0})

if "combined_text" in manual_df.columns:
    manual_df["model_text"] = manual_df["combined_text"].fillna("").astype(str).map(normalize_text)
elif "article_text" in manual_df.columns:
    manual_df["model_text"] = manual_df["article_text"].fillna("").astype(str).map(normalize_text)
elif "translated_text" in manual_df.columns:
    manual_df["model_text"] = manual_df["translated_text"].fillna("").astype(str).map(normalize_text)
else:
    manual_df["model_text"] = (
        manual_df.get("title", "").fillna("").astype(str)
        + "\n"
        + manual_df.get("body", "").fillna("").astype(str)
    ).map(normalize_text)

manual_df = manual_df[manual_df["model_text"].str.strip().ne("")].copy()
print(f"Training rows: {len(manual_df):,}")
display(manual_df["y"].value_counts().rename(index={0: "No", 1: "Political corruption"}))

Training rows: 452


y
No                      332
Political corruption    120
Name: count, dtype: int64

In [21]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=200_000,
            lowercase=True,
        ),
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42,
        ),
    ),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_true = manual_df["y"].values

y_pred = cross_val_predict(model, manual_df["model_text"], y_true, cv=cv, method="predict")

print(classification_report(
    y_true,
    y_pred,
    target_names=["No political corruption", "Political corruption"],
    zero_division=0,
))

display(pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=["True No", "True Political"],
    columns=["Pred No", "Pred Political"],
))


                         precision    recall  f1-score   support

No political corruption       0.83      0.75      0.79       332
   Political corruption       0.45      0.56      0.50       120

               accuracy                           0.70       452
              macro avg       0.64      0.66      0.64       452
           weighted avg       0.73      0.70      0.71       452



,Pred No,Pred Political
True No,250,82
True Political,53,67


In [22]:
y_prob = cross_val_predict(model, manual_df["model_text"], y_true, cv=cv, method="predict_proba")[:, 1]
manual_df["cv_prob_political_corruption"] = y_prob
manual_df["cv_pred"] = (manual_df["cv_prob_political_corruption"] >= 0.5).astype(int)

display(
    manual_df[
        ["country", "corruption_label_m", "cv_prob_political_corruption", "cv_pred", "model_text"]
    ].sort_values("cv_prob_political_corruption", ascending=False).head(20)
)


,country,corruption_label_m,cv_prob_political_corruption,cv_pred,model_text
721,Hungary,no political corruption,0.703764,1,Az igazságügyi miniszter hazugságai Azt gondol...
1297,Ukraine,no political corruption,0.700963,1,Зачем России почетное консульство Никарагуа в ...
1288,Ukraine,political corruption,0.686699,1,"""Імпічмент. Відставка. Суд"". Зеленський, Шефір..."
32,Bulgaria,no political corruption,0.683943,1,"Акция на ""Киберсигурност"" в Разлог и Перник, и..."
1430,Ukraine,no political corruption,0.683053,1,Депутата Яценка звинувачують у сексизмі: подро...
1299,Ukraine,political corruption,0.682299,1,Спростування Генштабу і температурні рекорди у...
1295,Ukraine,political corruption,0.682073,1,На Банковій задумали прирівняти масштабну кору...
683,Hungary,political corruption,0.676821,1,Nem szavazták meg a rokonok kizárását a hódmez...
1433,Ukraine,political corruption,0.671918,1,Справою про конфлікт інтересів прем'єра Чехії ...
11,Bulgaria,political corruption,0.670498,1,Прокуратурата разпитва Ивайла Бакалова и Вален...


In [23]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

X_text = manual_df["model_text"].tolist()
y_true = manual_df["y"].values

X_emb = embedder.encode(
    X_text,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embedding_clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

y_pred_emb = cross_val_predict(
    embedding_clf,
    X_emb,
    y_true,
    cv=cv,
    method="predict",
)

print(classification_report(
    y_true,
    y_pred_emb,
    target_names=["No political corruption", "Political corruption"],
    zero_division=0,
))

display(pd.DataFrame(
    confusion_matrix(y_true, y_pred_emb),
    index=["True No", "True Political"],
    columns=["Pred No", "Pred Political"],
))


                         precision    recall  f1-score   support

No political corruption       0.88      0.74      0.80       332
   Political corruption       0.50      0.72      0.59       120

               accuracy                           0.73       452
              macro avg       0.69      0.73      0.69       452
           weighted avg       0.78      0.73      0.74       452



,Pred No,Pred Political
True No,245,87
True Political,34,86


In [24]:
y_prob_emb = cross_val_predict(
    embedding_clf,
    X_emb,
    y_true,
    cv=cv,
    method="predict_proba",
)[:, 1]

manual_df["cv_prob_emb_political_corruption"] = y_prob_emb
manual_df["cv_pred_emb"] = (manual_df["cv_prob_emb_political_corruption"] >= 0.5).astype(int)
manual_df["emb_correct"] = manual_df["cv_pred_emb"].eq(manual_df["y"])

print("Accuracy by country:")
display(
    manual_df.groupby("country")["emb_correct"]
    .mean()
    .sort_values()
    .to_frame("embedding_accuracy")
)

false_positives_emb = manual_df[
    (manual_df["y"] == 0) & (manual_df["cv_pred_emb"] == 1)
].copy()

false_negatives_emb = manual_df[
    (manual_df["y"] == 1) & (manual_df["cv_pred_emb"] == 0)
].copy()

print(f"False positives: {len(false_positives_emb)}")
display(false_positives_emb[
    ["country", "corruption_label_m", "cv_prob_emb_political_corruption", "model_text"]
].sort_values("cv_prob_emb_political_corruption", ascending=False).head(20))

print(f"False negatives: {len(false_negatives_emb)}")
display(false_negatives_emb[
    ["country", "corruption_label_m", "cv_prob_emb_political_corruption", "model_text"]
].sort_values("cv_prob_emb_political_corruption", ascending=True).head(20))

Accuracy by country:


,embedding_accuracy
country,
Bulgaria,0.604167
Ukraine,0.604167
Serbia,0.653846
Hungary,0.725490
Sweden,0.745098
Italy,0.803922
United_Kingdom,0.803922
France,0.816327
Netherlands,0.823529


False positives: 87


,country,corruption_label_m,cv_prob_emb_political_corruption,model_text
721,Hungary,no political corruption,0.845019,Az igazságügyi miniszter hazugságai Azt gondol...
890,Netherlands,no political corruption,0.810823,Hooggerechtshof Venezuela doet omstreden aanst...
1313,Ukraine,no political corruption,0.810500,Глава СБУ Баканов потрапив під приціл НАБУ: у ...
794,Italy,no political corruption,0.799432,"CASO GILARDI, 3° CAPITOLO. FASCICOLO IN PROCUR..."
916,Netherlands,no political corruption,0.787130,Motief voor fraude blijft een raadsel Landsadv...
1429,Ukraine,no political corruption,0.768657,"""Це питання - справа честі"", - Гройсман про ро..."
1302,Ukraine,no political corruption,0.763616,Гетманцев назвав Податкову службу корупційним ...
928,Netherlands,no political corruption,0.762308,Meerdere meldingen van wangedrag door senaatsv...
725,Hungary,no political corruption,0.753701,"Ciolacu csomagja ""Félek a görögöktől, ha ajánd..."
1432,Ukraine,no political corruption,0.737331,Корупція в кар'єрі Павелко: Столична асоціація...


False negatives: 34


,country,corruption_label_m,cv_prob_emb_political_corruption,model_text
1215,Sweden,political corruption,0.235575,Våldsamt vid protester mot Honduras president ...
40,Bulgaria,political corruption,0.238230,Двама чужденци подхвърлиха 30 евро на полицай ...
1317,Ukraine,political corruption,0.288149,ÐÑÑÐ½Ð°Ð»ÑÑÑÐ¸ Ð¾Ð¿ÑÐ¸Ð»ÑÐ´Ð½Ð¸Ð»Ð¸ Ñ...
1315,Ukraine,political corruption,0.302098,Навальний повернувся в Росію: що відомо Російс...
1308,Ukraine,political corruption,0.321085,Масштабні протести в Румунії: на вулиці вийшли...
694,Hungary,political corruption,0.322143,A gázkamrázós Sorosozó Demeter Szilárd testvér...
884,Netherlands,political corruption,0.324199,Zomerinterview (5): Hoe filmmaker Roberto Hern...
16,Bulgaria,political corruption,0.329364,Борис Джонсън и Тръмп с таен пакт за свободна ...
3,Bulgaria,political corruption,0.335739,Трима задържани за измама с евросредства за зе...
924,Netherlands,political corruption,0.343073,"Coke, corruptie en criminelen in de Rotterdams..."


## Active-Learning Batch

This samples 5,000 cleaned articles per country, scores them with the multilingual embedding classifier, and creates a balanced annotation batch. The batch intentionally includes uncertain cases, likely positives, and likely negatives.


In [26]:
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

POOL_PER_COUNTRY = 5000

pool_parts = []

country_groups = list(df_clean.groupby("country", dropna=False))

for country, country_df in tqdm(country_groups, desc="Sampling active-learning pool by country"):
    n = min(len(country_df), POOL_PER_COUNTRY)
    sampled = country_df.sample(n=n, random_state=RANDOM_STATE).copy()
    pool_parts.append(sampled)

df_pool = pd.concat(pool_parts, ignore_index=True)

print(f"Active-learning pool size: {len(df_pool):,}")
display(df_pool["country"].value_counts().to_frame("pool_rows"))

Active-learning pool size: 45,000


,pool_rows
country,
Bulgaria,5000
France,5000
Hungary,5000
Italy,5000
Netherlands,5000
Serbia,5000
Sweden,5000
Ukraine,5000
United_Kingdom,5000


In [27]:
from tqdm.auto import tqdm
import numpy as np

# Fit on all current manual labels.
embedding_clf.fit(X_emb, y_true)

df_scored = df_pool.copy()

# This is much cheaper on 45k rows than on 2.26M rows.
tqdm.pandas(desc="Normalizing active-learning pool text")
df_scored["model_text"] = (
    df_scored["article_text"]
    .fillna("")
    .astype(str)
    .progress_map(normalize_text)
)

texts = df_scored["model_text"].tolist()

OUTER_BATCH_SIZE = 512
INNER_EMBED_BATCH_SIZE = 64

all_probs = []

batch_starts = list(range(0, len(texts), OUTER_BATCH_SIZE))

for start in tqdm(batch_starts, desc="Embedding + scoring active-learning pool"):
    end = min(start + OUTER_BATCH_SIZE, len(texts))
    batch_texts = texts[start:end]

    batch_emb = embedder.encode(
        batch_texts,
        batch_size=INNER_EMBED_BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
    )

    batch_probs = embedding_clf.predict_proba(batch_emb)[:, 1]
    all_probs.append(batch_probs)

df_scored["prob_political_corruption"] = np.concatenate(all_probs)
df_scored["model_pred"] = (
    df_scored["prob_political_corruption"] >= 0.5
).astype(int)

display(df_scored["prob_political_corruption"].describe())
print(f"Predicted positive rate: {df_scored['model_pred'].mean():.2%}")

print("Predicted positive rate by country:")
display(
    df_scored.groupby("country")["model_pred"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("predicted_positive_rate")
)

count    45000.000000
mean         0.422783
std          0.200917
min          0.018799
25%          0.259513
50%          0.417377
75%          0.580554
max          0.934570
Name: prob_political_corruption, dtype: float64

Predicted positive rate: 36.86%
Predicted positive rate by country:


,predicted_positive_rate
country,
Ukraine,0.6518
Bulgaria,0.5276
Serbia,0.4462
Hungary,0.4412
Italy,0.3054
France,0.3012
Netherlands,0.2842
Sweden,0.2366
United_Kingdom,0.1230


In [29]:
AL_OUTPUT_DIR = DEDUP_OUTPUT_DIR / "active_learning"
AL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_TARGETS = {
    "Ukraine": 150,
    "Bulgaria": 150,
    "Serbia": 150,
    "Hungary": 120,
    "Sweden": 100,
    "Italy": 80,
    "France": 80,
    "Netherlands": 80,
    "United_Kingdom": 80,
}

sample_cols = [
    "uri",
    "country",
    "date_parsed",
    "year",
    "source_uri",
    "article_text",
    "prob_political_corruption",
    "model_pred",
]

sample_cols = [col for col in sample_cols if col in df_scored.columns]

labeled_uris = set()
if "uri" in manual_df.columns:
    labeled_uris = set(manual_df["uri"].dropna().astype(str))

batches = []

for country, target_n in COUNTRY_TARGETS.items():
    country_df = df_scored[df_scored["country"] == country].copy()

    if "uri" in country_df.columns and labeled_uris:
        country_df = country_df[
            ~country_df["uri"].astype(str).isin(labeled_uris)
        ].copy()

    n_uncertain = int(target_n * 0.50)
    n_high_pos = int(target_n * 0.35)
    n_high_neg = target_n - n_uncertain - n_high_pos

    uncertain_pool = country_df[
        country_df["prob_political_corruption"].between(0.4, 0.6)
    ].copy()
    uncertain_pool["al_bucket"] = "uncertain"
    uncertain_pool["distance_to_0_5"] = (
        uncertain_pool["prob_political_corruption"] - 0.5
    ).abs()

    high_pos_pool = country_df[
        country_df["prob_political_corruption"] >= 0.7
    ].copy()
    high_pos_pool["al_bucket"] = "high_predicted_positive"

    high_neg_pool = country_df[
        country_df["prob_political_corruption"] <= 0.3
    ].copy()
    high_neg_pool["al_bucket"] = "high_predicted_negative"

    uncertain_sample = uncertain_pool.sort_values("distance_to_0_5").head(n_uncertain)

    high_pos_sample = high_pos_pool.sort_values(
        "prob_political_corruption",
        ascending=False,
    ).head(n_high_pos)

    high_neg_sample = high_neg_pool.sort_values(
        "prob_political_corruption",
        ascending=True,
    ).head(n_high_neg)

    country_batch = pd.concat(
        [uncertain_sample, high_pos_sample, high_neg_sample],
        ignore_index=True,
    )

    if len(country_batch) < target_n:
        selected_uris = set(country_batch["uri"].dropna().astype(str)) if "uri" in country_batch.columns else set()
        topup_pool = country_df[
            ~country_df["uri"].astype(str).isin(selected_uris)
        ].copy()
        topup_pool["al_bucket"] = "topup_uncertain"
        topup_pool["distance_to_0_5"] = (
            topup_pool["prob_political_corruption"] - 0.5
        ).abs()

        topup = topup_pool.sort_values("distance_to_0_5").head(target_n - len(country_batch))
        country_batch = pd.concat([country_batch, topup], ignore_index=True)

    batches.append(country_batch)

active_learning_batch = pd.concat(batches, ignore_index=True)

active_learning_batch["human_final_label"] = ""
active_learning_batch["human_notes"] = ""

output_cols = [
    "al_bucket",
    "uri",
    "country",
    "date_parsed",
    "year",
    "source_uri",
    "prob_political_corruption",
    "model_pred",
    "human_final_label",
    "human_notes",
    "article_text",
]

output_cols = [col for col in output_cols if col in active_learning_batch.columns]

active_learning_path = AL_OUTPUT_DIR / "active_learning_batch_for_annotation.csv"
active_learning_batch[output_cols].to_csv(active_learning_path, index=False)

print(f"Saved {len(active_learning_batch):,} rows to: {active_learning_path}")
display(active_learning_batch["al_bucket"].value_counts())
display(active_learning_batch.groupby(["country", "al_bucket"]).size().unstack(fill_value=0))

Saved 990 rows to: /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/active_learning/active_learning_batch_for_annotation.csv


al_bucket
uncertain                  495
high_predicted_positive    345
high_predicted_negative    150
Name: count, dtype: int64

al_bucket,high_predicted_negative,high_predicted_positive,uncertain
country,,,
Bulgaria,23,52,75
France,12,28,40
Hungary,18,42,60
Italy,12,28,40
Netherlands,12,28,40
Serbia,23,52,75
Sweden,15,35,50
Ukraine,23,52,75
United_Kingdom,12,28,40
